# **This notebook acts as the dataloader of eICU**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import seaborn as sns
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.metrics import mean_absolute_error, r2_score
import os
import glob

## **eICU**

In [ ]:
path_client_2 = "../../Datasets/eICU_data"

def load_csv(name):
    file_path = os.path.join(path_client_2, f"{name}.csv")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} not found.")
    print(f"Loading {file_path} ...")
    return pd.read_csv(file_path, low_memory=False)

dataframes = {}

dataframes["admissionDx"] = load_csv("admissionDx")
dataframes["patient"] = load_csv("patient")
dataframes["diagnosis"] = load_csv("diagnosis")
dataframes["lab"] = load_csv("lab")
dataframes["medication"] = load_csv("medication")
dataframes["infusionDrug"] = load_csv("infusionDrug")
dataframes["intakeOutput"] = load_csv("intakeOutput")
dataframes["microLab"] = load_csv("microLab")
dataframes["nurseAssessment"] = load_csv("nurseAssessment")
dataframes["respiratoryCharting"] = load_csv("respiratoryCharting")
dataframes["customLab"] = load_csv("customLab")
# dataframes["carePlanGeneral"] = load_csv("carePlanGeneral")
# dataframes["nurseCharting"] = load_csv("nurseCharting")

# print("Loaded:", list(dataframes.keys()))

In [ ]:
print(dataframes["patient"].columns.tolist())

In [ ]:
OBS_WINDOW = pd.Timedelta(hours=24)
BIN_SIZE = pd.Timedelta(hours=1)
id_col = "patientunitstayid"
client_2 = dataframes["patient"].copy()
 
print(f"Initial client_2 shape from patient table: {client_2.shape}")
print(f"Columns in client_2 after loading patient table: {client_2.columns.tolist()}")

if "unitadmittime24" in client_2.columns:
    client_2["unitadmittime"] = pd.to_datetime(client_2["unitadmittime24"], errors="coerce")
if "unitdischargetime24" in client_2.columns:
    client_2["unitdischtime"] = pd.to_datetime(client_2["unitdischargetime24"], errors="coerce")

stays_for_event_windowing = client_2[[id_col, "unitadmittime"]].copy()

In [ ]:
client_2.shape[1]

In [ ]:
dataframes["patient"]

In [ ]:
for name, df in dataframes.items():
    print(name, df.columns.tolist())

In [ ]:
merge_log = []

for name, df in dataframes.items():
    if name == "patient":
        continue
    if id_col not in df.columns:
        print(f"Skipping {name}: no {id_col}")
        continue

    print(f"\nProcessing {name}.csv")
    df = df.copy()

    # 1) Reconstruct eventtime from either datetime or offset
    dt_cols  = [c for c in df.columns if "datetime" in c.lower()]
    off_cols = [c for c in df.columns if "offset"   in c.lower()]

    if dt_cols:
        # use the first datetime column
        df[dt_cols[0]] = pd.to_datetime(df[dt_cols[0]], errors="coerce")
        df = df.merge(stays_for_event_windowing, on=id_col, how="left")
        df["eventtime"] = df[dt_cols[0]]

    elif off_cols:
        # use the first offset column (minutes since admit)
        df = df.merge(stays_for_event_windowing, on=id_col, how="left")
        df["eventtime"] = (
            df["unitadmittime"]
            + pd.to_timedelta(df[off_cols[0]].astype(float), unit="m")
        )

    else:
        print(f"  No datetime or offset in {name}, skipping")
        continue

    # 2) Filter to the first 24 h and assign hour bins 0…23
    before = len(df)
    df = df.loc[
        (df["eventtime"] >= df["unitadmittime"]) &
        (df["eventtime"] <  df["unitadmittime"] + OBS_WINDOW)
    ].copy()
    after = len(df)
    print(f"  Kept {after}/{before} rows in first 24 h")

    df["hour_from_intime"] = (
        (df["eventtime"] - df["unitadmittime"])
        .dt.total_seconds()
        .floordiv(BIN_SIZE.total_seconds())
        .astype(int)
    )

    # 3) Aggregate per (patientunitstayid, hour_from_intime)
    # 3a) Numeric stats
    numeric_cols = df.select_dtypes(include="number") \
                     .columns.difference([id_col, "hour_from_intime"])
    if len(numeric_cols):
        num_grp = df.groupby([id_col, "hour_from_intime"])[numeric_cols] \
                    .agg(["mean","std","min","max","count"])
        num_wide = num_grp.unstack(level="hour_from_intime", fill_value=np.nan)
        num_wide.columns = [
            f"{name}_{orig}_{stat}_h{hour}"
            for orig, stat, hour in num_wide.columns
        ]
        num_wide = num_wide.reset_index()
    else:
        num_wide = None

    # 3b) Categorical mode
    cat_cols = df.select_dtypes(include=["object","category","bool"]) \
                 .columns.difference([id_col, "hour_from_intime"])
    if len(cat_cols):
        cat_grp = df.groupby([id_col, "hour_from_intime"])[cat_cols] \
                    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
        cat_wide = cat_grp.unstack(level="hour_from_intime", fill_value=pd.NA)
        cat_wide.columns = [
            f"{name}_{orig}_mode_h{hour}"
            for orig, hour in cat_wide.columns
        ]
        cat_wide = cat_wide.reset_index()
    else:
        cat_wide = None

    # 4) Merge the numeric & categorical wide tables and drop all-NaN columns
    parts = [t for t in (num_wide, cat_wide) if t is not None]
    if not parts:
        print(f"  Skipping {name}: nothing to aggregate")
        continue

    hourly_wide = parts[0]
    for t in parts[1:]:
        hourly_wide = hourly_wide.merge(t, on=id_col, how="outer")

    keep_cols = [c for c in hourly_wide.columns
                 if c == id_col or not hourly_wide[c].isna().all()]
    hourly_wide = hourly_wide[keep_cols]

    # 5) Final merge into client_2
    client_2 = client_2.merge(hourly_wide, on=id_col, how="left")
    merge_log.append((name, "hourly"))

print("\nFinal shape:", client_2.shape)
print("Merge log:", merge_log)

In [ ]:
client_2.shape

In [ ]:
client_2

In [ ]:
# optional: save
client_2.to_csv("client_2_raw_hour.csv", index=False)
print("Saved client_2_raw_hour.csv")